# Smart MCQ Solver

This project predicts the **top 3 correct answers** for each multiple-choice question using several NLP approaches.

### Models Used
- TF-IDF + Cosine Similarity
- Sentence Transformers
- Zero-shot NLI
- Retrieval-Augmented Ranking (RAG)
- Multi-Layer Perceptron (MLP)
- Ensemble Model

**Evaluation Metric:** MAP@3

## 1. Setup

In [1]:
# Install required packages
!pip install -q wandb sentence-transformers

import os
import re
import json
import warnings

import numpy as np
import pandas as pd
import wandb

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")
print(f"Pandas: {pd.__version__}")
print(f"NumPy : {np.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.4 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
from kaggle_secrets import UserSecretsClient

# Login to Weights & Biases
secrets = UserSecretsClient()
WANDB_KEY = secrets.get_secret("WandB_API_key")

wandb.login(key=WANDB_KEY, relogin=True)

print("W&B login successful.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: soumyaranjanpanda01 (23f2004742-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful.


In [3]:
# Paths
BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"

TRAIN_PATH = f"{BASE}/train.csv"
TEST_PATH = f"{BASE}/test.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"

# Constants
OPTIONS = ["A", "B", "C", "D", "E"]
SEED = 42
VAL_SIZE = 0.20

WANDB_PROJECT = "23f2004742-t22026"

## 2. Load Dataset

In [4]:
# Load datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train Shape : {train_df.shape}")
print(f"Test Shape  : {test_df.shape}")

print("\nTraining Answer Distribution")
print(train_df["answer"].value_counts().sort_index())

train_df.head(3)

Train Shape : (2000, 8)
Test Shape  : (500, 7)

Training Answer Distribution
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
# Quick dataset overview

# Option length statistics
for opt in OPTIONS:
    lengths = train_df[opt].str.len()
    print(f"{opt}: Mean = {lengths.mean():.0f}, Max = {lengths.max()}")

# Duplicate prompts
prompt_counts = train_df["prompt"].value_counts()

print(f"\nUnique Prompts : {len(prompt_counts)}")
print(f"Repeated Prompts: {(prompt_counts > 1).sum()}")

# Question pattern
def get_style(prompt):
    prompt = prompt.lower()

    if "pick the best" in prompt:
        return "Pick the best"
    if "choose the correct" in prompt:
        return "Choose the correct"
    if "which of the following" in prompt:
        return "Which of the following"

    return "Other"

print("\nQuestion Styles")
print(train_df["prompt"].apply(get_style).value_counts())

A: Mean = 164, Max = 472
B: Mean = 167, Max = 662
C: Mean = 167, Max = 530
D: Mean = 163, Max = 450
E: Mean = 164, Max = 587

Unique Prompts : 1758
Repeated Prompts: 212

Question Styles
prompt
Other                     1063
Which of the following     329
Pick the best              316
Choose the correct         292
Name: count, dtype: int64


## 3. Train / Validation Split

Split the dataset by **prompt** so that the same question never appears in both training and validation sets.

In [6]:
# Split unique prompts
unique_prompts = train_df["prompt"].unique()

train_prompts, val_prompts = train_test_split(
    unique_prompts,
    test_size=VAL_SIZE,
    random_state=SEED
)

train_split = (
    train_df[train_df["prompt"].isin(train_prompts)]
    .reset_index(drop=True)
)

val_split = (
    train_df[train_df["prompt"].isin(val_prompts)]
    .reset_index(drop=True)
)

# Ensure no prompt leakage
overlap = set(train_split["prompt"]) & set(val_split["prompt"])
assert len(overlap) == 0

print(f"Train : {len(train_split)} rows ({len(train_split)/len(train_df):.1%})")
print(f"Valid : {len(val_split)} rows ({len(val_split)/len(train_df):.1%})")
print(f"Prompt Overlap : {len(overlap)} ✓")

Train : 1592 rows (79.6%)
Valid : 408 rows (20.4%)
Prompt Overlap : 0 ✓


## 4. Text Preprocessing

In [7]:
def clean_text(text: str) -> str:
    """Clean prompts and options."""

    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    boilerplate = [
        r"^Pick the best possible answer:\s*",
        r"^Choose the correct answer:\s*",
        r"^Select the most accurate option:\s*",
        r"^Identify the correct statement:\s*",
        r"^Determine the correct option:\s*",
        r"\s*among the listed options\.?$",
        r"\s*from the following choices\.?$",
        r"\s*based on the given context\.?$",
        r"\s*carefully\.?$",
    ]

    for pattern in boilerplate:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    return text


def preprocess_row(row):
    """Return a cleaned question."""

    return {
        "prompt": clean_text(row["prompt"]),
        "A": clean_text(row["A"]),
        "B": clean_text(row["B"]),
        "C": clean_text(row["C"]),
        "D": clean_text(row["D"]),
        "E": clean_text(row["E"]),
    }


# Apply preprocessing
train_split = train_split.copy()
val_split = val_split.copy()
test_df_clean = test_df.copy()

for df in [train_split, val_split, test_df_clean]:
    df["prompt"] = df["prompt"].apply(clean_text)

    for opt in OPTIONS:
        df[opt] = df[opt].apply(clean_text)

print("Text preprocessing completed.")

raw = train_df.iloc[0]["prompt"]
clean = train_split.iloc[0]["prompt"]

print("\nBefore:", raw[:100])
print("After :", clean[:100])

Text preprocessing completed.

Before: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and 
After : What is Martin Heidegger's view on the relationship between time and human existence?


## 5. Evaluation Metric (MAP@3)

In [8]:
def average_precision_at_3(actual, predicted):
    """Compute AP@3 for one prediction."""

    predicted = [str(p).strip().upper() for p in predicted[:3]]

    for rank, pred in enumerate(predicted, start=1):
        if pred == actual:
            return 1 / rank

    return 0.0


def map_at_3(actuals, predictions):
    """Compute Mean Average Precision@3."""

    assert len(actuals) == len(predictions)

    scores = [
        average_precision_at_3(a, p)
        for a, p in zip(actuals, predictions)
    ]

    return float(np.mean(scores))


# Quick verification
assert average_precision_at_3("A", ["A", "B", "C"]) == 1
assert average_precision_at_3("A", ["B", "A", "C"]) == 0.5
assert abs(average_precision_at_3("A", ["C", "D", "A"]) - 1 / 3) < 1e-9
assert average_precision_at_3("A", ["B", "C", "D"]) == 0

print("MAP@3 verified ✓")

MAP@3 verified ✓


## 6. TF-IDF Baseline

Create a TF-IDF index using the training set's correct answers and rank each option by cosine similarity.

In [9]:
# Build TF-IDF index
correct_texts_train = [
    f"{row['prompt']} {row[row['answer']]}"
    for _, row in train_split.iterrows()
]

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=80000,
    strip_accents="unicode",
    min_df=1,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(correct_texts_train)

print(f"Vocabulary Size : {len(tfidf_vectorizer.vocabulary_)}")
print(f"Matrix Shape    : {tfidf_matrix.shape}")

Vocabulary Size : 7733
Matrix Shape    : (1592, 7733)


In [10]:
def tfidf_rank_options(row):
    """Rank answer options using TF-IDF cosine similarity."""

    scores = {}

    for opt in OPTIONS:
        query = f"{row['prompt']} {row[opt]}"
        query_vec = tfidf_vectorizer.transform([query])

        scores[opt] = float(
            cosine_similarity(query_vec, tfidf_matrix).max()
        )

    return sorted(scores, key=lambda x: -scores[x])


# Validation
tfidf_predictions = [
    tfidf_rank_options(row)[:3]
    for _, row in val_split.iterrows()
]

tfidf_map3 = map_at_3(
    val_split["answer"].tolist(),
    tfidf_predictions,
)

top1 = sum(
    pred[0] == ans
    for pred, ans in zip(tfidf_predictions, val_split["answer"])
)

print(f"MAP@3   : {tfidf_map3:.4f}")
print(f"Top-1   : {top1}/{len(val_split)} ({top1/len(val_split):.2%})")

MAP@3   : 0.9890
Top-1   : 401/408 (98.28%)


In [11]:
# Log TF-IDF results
wandb.init(
    project=WANDB_PROJECT,
    name="tfidf-baseline",
    config={
        "model": "TF-IDF Cosine Similarity",
        "ngram_range": "(1,2)",
        "max_features": 80000,
        "sublinear_tf": True,
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

wandb.log({
    "val_map3": tfidf_map3,
    "val_top1_acc": top1 / len(val_split),
})

wandb.finish()

print("TF-IDF results logged.")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


val_map3,▁
val_top1_acc,▁
val_map3,0.98897
val_top1_acc,0.98284


TF-IDF results logged.


## 7. Word2Vec

Train a Word2Vec model on the training corpus and represent each question-answer pair by the average of its word embeddings. Candidate options are ranked using cosine similarity.

In [12]:
!pip install -q gensim

In [13]:
from gensim.models import Word2Vec
from sklearn.preprocessing import normalize

In [14]:
# Prepare corpus for Word2Vec

train_corpus = []

for _, row in train_split.iterrows():

    sentence = (
        f"{row['prompt']} "
        f"{row[row['answer']]}"
    )

    train_corpus.append(
        clean_text(sentence).lower().split()
    )

print(f"Training sentences : {len(train_corpus)}")

Training sentences : 1592


In [15]:
print("Training Word2Vec...")

w2v_model = Word2Vec(
    sentences=train_corpus,
    vector_size=300,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=50,
    seed=SEED
)

print("Vocabulary size :", len(w2v_model.wv))

Training Word2Vec...
Vocabulary size : 2562


In [16]:
def sentence_embedding(text):

    words = clean_text(text).lower().split()

    vectors = [
        w2v_model.wv[word]
        for word in words
        if word in w2v_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(w2v_model.vector_size)

    return np.mean(vectors, axis=0)

In [17]:
print("Encoding training corpus...")

w2v_train_embeddings = np.array([
    sentence_embedding(
        f"{row['prompt']} {row[row['answer']]}"
    )
    for _, row in train_split.iterrows()
])

w2v_train_embeddings = normalize(
    w2v_train_embeddings
)

print(w2v_train_embeddings.shape)

Encoding training corpus...
(1592, 300)


In [18]:
def w2v_rank_options(row):

    option_embeddings = np.array([

        sentence_embedding(
            f"{row['prompt']} {row[opt]}"
        )

        for opt in OPTIONS

    ])

    option_embeddings = normalize(option_embeddings)

    similarities = (
        option_embeddings @
        w2v_train_embeddings.T
    ).max(axis=1)

    return [
        OPTIONS[i]
        for i in np.argsort(-similarities)
    ]

In [19]:
print("Evaluating Word2Vec...")

w2v_predictions = [

    w2v_rank_options(row)[:3]

    for _, row in val_split.iterrows()

]

w2v_map3 = map_at_3(
    val_split["answer"].tolist(),
    w2v_predictions
)

top1_w2v = sum(

    pred[0] == ans

    for pred, ans in zip(
        w2v_predictions,
        val_split["answer"]
    )

)

print(f"MAP@3 : {w2v_map3:.4f}")
print(
    f"Top-1 : {top1_w2v}/{len(val_split)} "
    f"({top1_w2v/len(val_split):.2%})"
)

Evaluating Word2Vec...
MAP@3 : 0.9963
Top-1 : 405/408 (99.26%)


In [20]:
wandb.init(

    project=WANDB_PROJECT,

    name="word2vec",

    config={

        "model": "Word2Vec",

        "vector_size": 300,

        "window": 5,

        "epochs": 50,

        "sg": 1,

        "val_size": VAL_SIZE,

    },

    reinit=True

)

wandb.log({

    "val_map3": w2v_map3,

    "val_top1_acc": top1_w2v / len(val_split)

})

wandb.finish()

print("Word2Vec results logged.")

val_map3,▁
val_top1_acc,▁
val_map3,0.99632
val_top1_acc,0.99265


Word2Vec results logged.


In [51]:
print(w2v_map3)

0.9963235294117647


## 8. Sentence Transformer

Use a pretrained sentence embedding model to compare the semantic similarity between questions and answer options.

In [21]:
from sentence_transformers import SentenceTransformer

print("Loading model...")

st_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded.")
print(f"Embedding Size : {st_model.get_sentence_embedding_dimension()}")

Loading model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.
Embedding Size : 384


In [22]:
# Encode training corpus
train_corpus = [
    f"{row['prompt']} {row[row['answer']]}"
    for _, row in train_split.iterrows()
]

print("Encoding training corpus...")

train_embeddings = st_model.encode(
    train_corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

train_embeddings = normalize(train_embeddings)

print(f"Embedding Matrix : {train_embeddings.shape}")

Encoding training corpus...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Embedding Matrix : (1592, 384)


In [23]:
def st_rank_options(row: pd.Series) -> list:
    """Rank answer options using sentence embedding similarity."""

    option_texts = [
        f"{row['prompt']} {row[opt]}"
        for opt in OPTIONS
    ]

    option_embs = st_model.encode(
        option_texts,
        convert_to_numpy=True
    )
    option_embs = normalize(option_embs)

    scores = (option_embs @ train_embeddings.T).max(axis=1)

    return [OPTIONS[i] for i in np.argsort(-scores)]


print("Evaluating on validation set...")

st_predictions = [
    st_rank_options(row)[:3]
    for _, row in val_split.iterrows()
]

st_map3 = map_at_3(
    val_split["answer"].tolist(),
    st_predictions
)

top1_st = sum(
    pred[0] == ans
    for pred, ans in zip(st_predictions, val_split["answer"])
)

print(f"MAP@3 : {st_map3:.4f}")
print(f"Top-1 : {top1_st}/{len(val_split)} ({top1_st/len(val_split):.2%})")

Evaluating on validation set...
MAP@3 : 0.9963
Top-1 : 405/408 (99.26%)


In [24]:
wandb.init(
    project=WANDB_PROJECT,
    name="sentence-transformer",
    config={
        "model": "all-MiniLM-L6-v2",
        "embedding_dim": st_model.get_sentence_embedding_dimension(),
        "similarity": "cosine",
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

wandb.log({
    "val_map3": st_map3,
    "val_top1_acc": top1_st / len(val_split),
})

wandb.finish()

print("Sentence Transformer results logged.")

val_map3,▁
val_top1_acc,▁
val_map3,0.99632
val_top1_acc,0.99265


Sentence Transformer results logged.


## 9. Zero-shot NLI

Use a pretrained Natural Language Inference (NLI) model to rank answer options based on how well they match the question.

In [25]:
from transformers import pipeline

print("Loading NLI model...")

nli_pipeline = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=0 if __import__("torch").cuda.is_available() else -1,
)

print("Model loaded.")

Loading NLI model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Model loaded.


In [26]:
def nli_rank_options(row: pd.Series) -> list:
    """Rank answer options using zero-shot NLI."""

    candidate_labels = [row[opt] for opt in OPTIONS]

    result = nli_pipeline(
        row["prompt"],
        candidate_labels=candidate_labels,
        multi_label=False,
    )

    label_to_option = {
        row[opt]: opt
        for opt in OPTIONS
    }

    return [
        label_to_option[label]
        for label in result["labels"]
    ]


print("Evaluating on validation set...")

NLI_SAMPLE = len(val_split)
val_sample = val_split.iloc[:NLI_SAMPLE]

nli_predictions = [
    nli_rank_options(row)[:3]
    for _, row in val_sample.iterrows()
]

nli_map3 = map_at_3(
    val_sample["answer"].tolist(),
    nli_predictions
)

top1_nli = sum(
    pred[0] == ans
    for pred, ans in zip(nli_predictions, val_sample["answer"])
)

print(f"MAP@3 : {nli_map3:.4f}")
print(f"Top-1 : {top1_nli}/{len(val_sample)} ({top1_nli/len(val_sample):.2%})")

Evaluating on validation set...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


MAP@3 : 0.5261
Top-1 : 159/408 (38.97%)


In [27]:
wandb.init(
    project=WANDB_PROJECT,
    name="zero-shot-nli",
    config={
        "model": "cross-encoder/nli-deberta-v3-small",
        "approach": "zero-shot classification",
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

wandb.log({
    "val_map3": nli_map3,
    "val_top1_acc": top1_nli / len(val_sample),
})

wandb.finish()

print("Zero-shot NLI results logged.")

val_map3,▁
val_top1_acc,▁
val_map3,0.52614
val_top1_acc,0.38971


Zero-shot NLI results logged.


## 10. Retrieval-Augmented Ranking (RAG)

Retrieve the most similar training questions and use their correct answers as additional context when ranking the options.

In [28]:
def build_rag_index(train_df: pd.DataFrame, encoder):
    """Build embeddings for training prompts."""

    prompts = train_df["prompt"].tolist()
    correct_answers = [
        row[row["answer"]]
        for _, row in train_df.iterrows()
    ]

    prompt_embs = encoder.encode(
        prompts,
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    prompt_embs = normalize(prompt_embs)

    return prompt_embs, correct_answers


print("Building retrieval index...")

rag_prompt_embs, rag_correct_texts = build_rag_index(
    train_split,
    st_model,
)

print(f"Index Size : {len(rag_prompt_embs)}")

Building retrieval index...
Index Size : 1592


In [29]:
def rag_rank_options(row: pd.Series, k: int = 5) -> list:
    """Rank answer options using retrieval-augmented similarity."""
    
    # Retrieve similar questions
    q_emb = st_model.encode([row['prompt']], convert_to_numpy=True)
    q_emb = normalize(q_emb)
    retrieval_scores = (q_emb @ rag_prompt_embs.T)[0]
    top_k_idx = np.argsort(-retrieval_scores)[:k]

    # Build retrieval context
    context_texts = [rag_correct_texts[i] for i in top_k_idx]
    context_embs  = st_model.encode(context_texts, convert_to_numpy=True)
    context_embs  = normalize(context_embs)

    # Score answer options
    option_texts = [f"{row['prompt']} {row[opt]}" for opt in OPTIONS]
    option_embs  = normalize(st_model.encode(option_texts, convert_to_numpy=True))

    sim_corpus  = (option_embs @ train_embeddings.T).max(axis=1)
    sim_context = (option_embs @ context_embs.T).max(axis=1)

    # Combine: give slightly more weight to retrieved context
    combined = 0.6 * sim_corpus + 0.4 * sim_context
    return [OPTIONS[i] for i in np.argsort(-combined)]


print("Evaluating RAG on validation set...")
rag_predictions = [rag_rank_options(row)[:3] for _, row in val_split.iterrows()]
rag_map3 = map_at_3(val_split['answer'].tolist(), rag_predictions)

top1_rag = sum(p[0] == a for p, a in zip(rag_predictions, val_split['answer']))
print(f"MAP@3 : {rag_map3:.4f}")
print(f"Top-1 : {top1_rag}/{len(val_split)} ({top1_rag/len(val_split):.2%})")

Evaluating RAG on validation set...
MAP@3 : 0.9690
Top-1 : 384/408 (94.12%)


In [30]:
wandb.init(
    project=WANDB_PROJECT,
    name="rag-retrieval",
    config={
        "model": "all-MiniLM-L6-v2",
        "approach": "RAG",
        "retrieval_k": 5,
        "corpus_weight": 0.6,
        "context_weight": 0.4,
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

wandb.log({
    "val_map3": rag_map3,
    "val_top1_acc": top1_rag / len(val_split),
})

wandb.finish()

print("RAG results logged.")

val_map3,▁
val_top1_acc,▁
val_map3,0.96895
val_top1_acc,0.94118


RAG results logged.


## 11. MLP from Scratch

Train a two-layer MLP using handcrafted TF-IDF similarity features to predict the correct answer option.

In [31]:
# Feature extraction
def extract_features(df: pd.DataFrame, vec, mat) -> tuple:
    """Extract TF-IDF similarity features for the MLP."""
    
    X, y = [], []
    for _, row in df.iterrows():
        sims = []
        for opt in OPTIONS:
            q = f"{row['prompt']} {row[opt]}"
            v = vec.transform([q])
            sims.append(float(cosine_similarity(v, mat).max()))

        diffs = []
        for i in range(5):
            for j in range(i + 1, 5):
                diffs.append(sims[i] - sims[j])

        X.append(sims + diffs)
        if 'answer' in df.columns:
            y.append(OPTIONS.index(row['answer']))

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


print("Extracting features for train split...")
X_train, y_train = extract_features(train_split, tfidf_vectorizer, tfidf_matrix)
print("Extracting features for val split...")
X_val, y_val = extract_features(val_split, tfidf_vectorizer, tfidf_matrix)

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")

Extracting features for train split...
Extracting features for val split...
X_train : (1592, 15)
X_val   : (408, 15)


In [32]:
#  MLP in pure NumPy
class MLP:
    """Two-layer MLP implemented in NumPy."""
    
    def __init__(self, n_in, n_hidden, n_out, lr=0.05, seed=42):
        rng = np.random.default_rng(seed)
        scale = np.sqrt(2.0 / n_in)   # He initialisation
        self.W1 = rng.normal(0, scale, (n_in, n_hidden)).astype(np.float32)
        self.b1 = np.zeros(n_hidden, dtype=np.float32)
        self.W2 = rng.normal(0, np.sqrt(2.0 / n_hidden), (n_hidden, n_out)).astype(np.float32)
        self.b2 = np.zeros(n_out, dtype=np.float32)
        self.lr = lr

    def _relu(self, x):
        return np.maximum(0, x)

    def _softmax(self, x):
        ex = np.exp(x - x.max(axis=1, keepdims=True))
        return ex / ex.sum(axis=1, keepdims=True)

    def forward(self, X):
        self.h1  = self._relu(X @ self.W1 + self.b1)
        self.out = self._softmax(self.h1 @ self.W2 + self.b2)
        return self.out

    def backward(self, X, y_true):
        n = len(X)
        # Cross-entropy gradient w.r.t. softmax output
        d_out = self.out.copy()
        d_out[np.arange(n), y_true] -= 1
        d_out /= n
        # Layer 2 gradients
        dW2 = self.h1.T @ d_out
        db2 = d_out.sum(axis=0)
        # Layer 1 gradients (ReLU derivative = 1 if h1 > 0)
        d_h1 = (d_out @ self.W2.T) * (self.h1 > 0)
        dW1  = X.T @ d_h1
        db1  = d_h1.sum(axis=0)
        # Update weights
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def fit(self, X, y, epochs=300, batch_size=128, verbose=True):
        history = []
        n = len(X)
        for epoch in range(1, epochs + 1):
            idx = np.random.permutation(n)
            epoch_loss = 0.0
            for start in range(0, n, batch_size):
                batch = idx[start:start + batch_size]
                xb, yb = X[batch], y[batch]
                probs = self.forward(xb)
                loss  = -np.log(probs[np.arange(len(yb)), yb] + 1e-9).mean()
                epoch_loss += loss * len(yb)
                self.backward(xb, yb)
            avg_loss = epoch_loss / n
            history.append(avg_loss)
            if verbose and epoch % 50 == 0:
                print(f"  Epoch {epoch:3d}/{epochs} — loss: {avg_loss:.4f}")
        return history

    def predict_proba(self, X):
        return self.forward(X)


#  Train 
print("Training MLP from scratch...")
mlp = MLP(n_in=15, n_hidden=64, n_out=5, lr=0.05, seed=SEED)
history = mlp.fit(X_train, y_train, epochs=300, batch_size=128, verbose=True)

Training MLP from scratch...
  Epoch  50/300 — loss: 0.4000
  Epoch 100/300 — loss: 0.2509
  Epoch 150/300 — loss: 0.1849
  Epoch 200/300 — loss: 0.1461
  Epoch 250/300 — loss: 0.1206
  Epoch 300/300 — loss: 0.1028


In [33]:
# Evaluate on validation set 
proba_val = mlp.predict_proba(X_val)
mlp_preds = [[OPTIONS[j] for j in np.argsort(-proba_val[i])][:3]
             for i in range(len(X_val))]
mlp_map3  = map_at_3(val_split['answer'].tolist(), mlp_preds)
top1_mlp  = sum(p[0] == OPTIONS[y_val[i]] for i, p in enumerate(mlp_preds))

print(f"MAP@3 : {mlp_map3:.4f}")
print(f"Top-1 : {top1_mlp}/{len(val_split)} ({top1_mlp/len(val_split):.2%})")

# Log to W&B
wandb.init(
    project=WANDB_PROJECT,
    name="mlp-from-scratch",
    config={
        "model":        "MLP (pure NumPy)",
        "architecture": "15 → 64 → 5",
        "features":     "TF-IDF sim scores + pairwise diffs",
        "activation":   "ReLU + Softmax",
        "optimizer":    "mini-batch SGD",
        "epochs":       300,
        "batch_size":   128,
        "lr":           0.05,
        "val_size":     VAL_SIZE,
    },
    reinit=True
)
for i, loss in enumerate(history, 1):
    wandb.log({"epoch": i, "train_loss": loss})
wandb.log({"val_map3": mlp_map3, "val_top1_acc": top1_mlp / len(val_split)})
wandb.finish()
print("MLP run logged ✓")

MAP@3 : 0.9975
Top-1 : 406/408 (99.51%)


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇████
train_loss,█▇▅▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map3,▁
val_top1_acc,▁
epoch,300
train_loss,0.10283
val_map3,0.99755
val_top1_acc,0.9951


MLP run logged ✓


## 12. LoRA Fine-tuning

Fine-tune a pretrained transformer using **Low-Rank Adaptation (LoRA)** for binary sequence classification. Each `(question, option)` pair is treated as one training example, and the model ranks all five options using their predicted probabilities.

In [34]:
!pip install -q peft datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.3 MB/s eta 0:00:00:00:0100:01


In [35]:
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

import torch

In [36]:
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer loaded.


In [37]:
def build_lora_dataset(df):

    texts = []
    labels = []

    for _, row in df.iterrows():

        correct = row["answer"]

        for opt in OPTIONS:

            texts.append(
                f"Question: {row['prompt']}\n\n"
                f"Option: {row[opt]}"
            )

            labels.append(
                1 if opt == correct else 0
            )

    return Dataset.from_dict({
        "text": texts,
        "label": labels,
    })


train_dataset = build_lora_dataset(train_split)
val_dataset = build_lora_dataset(val_split)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 7960
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2040
})


In [38]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )


train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label",
    ],
)

val_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label",
    ],
)

print("Datasets ready.")

Map:   0%|          | 0/7960 [00:00<?, ? examples/s]

Map:   0%|          | 0/2040 [00:00<?, ? examples/s]

Datasets ready.


### Load DeBERTa and Apply LoRA

In [39]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Base model loaded.")

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight     

Base model loaded.


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [40]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 148,994 || all params: 142,045,444 || trainable%: 0.1049


In [41]:
training_args = TrainingArguments(
    output_dir="./lora_output",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=50,

    load_best_model_at_end=True,

    report_to="wandb",

    run_name="lora-deberta",

    seed=SEED,
)

In [42]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [43]:
print("Training LoRA model...")

trainer.train()

print("Training completed.")

Training LoRA model...


Epoch,Training Loss,Validation Loss
1,1.046139,0.984461
2,0.997802,0.969492
3,0.982247,0.953807


Training completed.


### Validation

In [44]:
def lora_option_score(prompt, option):

    text = (
        f"Question: {prompt}\n\n"
        f"Option: {option}"
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        logits = model(**inputs).logits

    probability = torch.softmax(logits, dim=-1)[0, 1]

    return probability.item()

In [45]:
def lora_rank_options(row):

    scores = {}

    for opt in OPTIONS:

        scores[opt] = lora_option_score(
            row["prompt"],
            row[opt],
        )

    return sorted(
        scores,
        key=lambda x: -scores[x]
    )

In [46]:
print("Evaluating LoRA...")

lora_predictions = [

    lora_rank_options(row)[:3]

    for _, row in val_split.iterrows()

]

lora_map3 = map_at_3(
    val_split["answer"].tolist(),
    lora_predictions,
)

top1_lora = sum(

    pred[0] == ans

    for pred, ans in zip(
        lora_predictions,
        val_split["answer"]
    )

)

print(f"MAP@3 : {lora_map3:.4f}")
print(
    f"Top-1 : "
    f"{top1_lora}/{len(val_split)} "
    f"({top1_lora/len(val_split):.2%})"
)

Evaluating LoRA...
MAP@3 : 0.6601
Top-1 : 216/408 (52.94%)


In [47]:
wandb.init(

    project=WANDB_PROJECT,

    name="lora-deberta",

    config={

        "model": MODEL_NAME,

        "method": "LoRA",

        "rank": 8,

        "alpha": 16,

        "epochs": 3,

        "learning_rate": 2e-4,

        "val_size": VAL_SIZE,

    },

    reinit=True,

)

wandb.log({

    "val_map3": lora_map3,

    "val_top1_acc": top1_lora / len(val_split),

})

wandb.finish()

print("LoRA results logged.")

eval/loss,█▅▁
eval/runtime,▁▆█
eval/samples_per_second,█▃▁
eval/steps_per_second,█▃▁
train/epoch,▁▂▂▃▃▃▄▄▅▅▅▆▆▇▇███
train/global_step,▁▂▂▃▃▃▄▄▅▅▅▆▆▇▇███
train/grad_norm,▂▂▇▂▃█▂▄▁▆▂▃▄▁
train/learning_rate,█▇▇▆▆▅▅▄▄▃▃▂▂▁
train/loss,█▃▁▇▄▄▄▃▄▂▁▄▂▃
eval/loss,0.95381
eval/runtime,4.6169


val_map3,▁
val_top1_acc,▁
val_map3,0.66013
val_top1_acc,0.52941


LoRA results logged.


In [48]:
print(lora_map3)

0.6601307189542484


## 13. Ensemble

Combine predictions from all models using Borda Count voting to produce the final top-3 ranked answers.

In [ ]:
def borda_ensemble(rankings: list) -> list:
    """Combine multiple rankings using the Borda count."""

    scores = {opt: 0 for opt in OPTIONS}
    n = len(OPTIONS)

    for ranking in rankings:
        for rank, opt in enumerate(ranking):
            scores[opt] += n - rank

    return sorted(scores, key=lambda opt: -scores[opt])


def ensemble_rank(
    row: pd.Series,
    tfidf_weight: float = 1.0,
    st_weight: float = 1.5,
    mlp_weight: float = 1.0,
) -> list:
    """Generate a weighted ensemble ranking."""

    tfidf_r = tfidf_rank_options(row)
    st_r = st_rank_options(row)

    sims = []
    for opt in OPTIONS:
        query = f"{row['prompt']} {row[opt]}"
        vec = tfidf_vectorizer.transform([query])
        sims.append(float(cosine_similarity(vec, tfidf_matrix).max()))

    diffs = [
        sims[i] - sims[j]
        for i in range(5)
        for j in range(i + 1, 5)
    ]

    x = np.array(sims + diffs, dtype=np.float32).reshape(1, -1)
    proba = mlp.predict_proba(x)[0]
    mlp_r = [OPTIONS[i] for i in np.argsort(-proba)]

    rankings = (
        [tfidf_r] * int(tfidf_weight * 2)
        + [st_r] * int(st_weight * 2)
        + [mlp_r] * int(mlp_weight * 2)
    )

    return borda_ensemble(rankings)


print("Evaluating ensemble...")

ens_predictions = [
    ensemble_rank(row)[:3]
    for _, row in val_split.iterrows()
]

ens_map3 = map_at_3(
    val_split["answer"].tolist(),
    ens_predictions,
)

top1_ens = sum(
    pred[0] == ans
    for pred, ans in zip(ens_predictions, val_split["answer"])
)

print(f"MAP@3 : {ens_map3:.4f}")
print(f"Top-1 : {top1_ens}/{len(val_split)} ({top1_ens/len(val_split):.2%})")

In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name="borda-ensemble",
    config={
        "model": "Borda Count Ensemble",
        "components": [
            "TF-IDF",
            "SentenceTransformer",
            "MLP-scratch",
        ],
        "weights": {
            "tfidf": 1.0,
            "st": 1.5,
            "mlp": 1.0,
        },
        "val_size": VAL_SIZE,
    },
    reinit=True,
)

wandb.log({
    "val_map3": ens_map3,
    "val_top1_acc": top1_ens / len(val_split),
})

wandb.finish()

print("Ensemble results logged.")

## 14. Model Comparison

In [ ]:
results = {
    "TF-IDF (M1)": tfidf_map3,
    "Sentence Transformer (M2)": st_map3,
    "Zero-shot NLI (M2)": nli_map3,
    "RAG (M3)": rag_map3,
    "MLP (M4)": mlp_map3,
    "Ensemble (M5)": ens_map3,
    "LoRA (M5)": lora_map3,
}

print(f"{'Model':<30}{'MAP@3':>10}")
print("-" * 42)

for model, score in results.items():
    print(f"{model:<30}{score:>10.4f}")

best_model = max(results, key=results.get)

print(f"\nBest Model: {best_model}")
print(f"MAP@3     : {results[best_model]:.4f}")

## 14. Generate Submission

In [ ]:
# Retrain all models using the complete training set.

print("Training TF-IDF...")
full_correct_texts = [
    f"{row['prompt']} {row[row['answer']]}"
    for _, row in train_df.iterrows()
]
tfidf_vectorizer_full = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True,
    max_features=80000, strip_accents="unicode"
)
tfidf_matrix_full = tfidf_vectorizer_full.fit_transform(full_correct_texts)

print("Encoding sentence embeddings...")
train_embeddings_full = normalize(
    st_model.encode(
        full_correct_texts, batch_size=64,
        show_progress_bar=True, convert_to_numpy=True
    )
)

print("Extracting MLP features...")
X_full, y_full = extract_features(train_df, tfidf_vectorizer_full, tfidf_matrix_full)

print("Training MLP...")
mlp_full = MLP(n_in=15, n_hidden=64, n_out=5, lr=0.05, seed=SEED)
mlp_full.fit(X_full, y_full, epochs=300, batch_size=128, verbose=False)
print("Done.")

In [ ]:
def predict_test_row(row: pd.Series) -> list:
    """Predict one test example using the ensemble."""

    # TF-IDF
    scores_tf = {}
    for opt in OPTIONS:
        q = f"{row['prompt']} {row[opt]}"
        v = tfidf_vectorizer_full.transform([q])
        scores_tf[opt] = float(cosine_similarity(v, tfidf_matrix_full).max())
    tfidf_r = sorted(scores_tf, key=lambda o: -scores_tf[o])

    # Sentence Transformer
    opt_texts = [f"{row['prompt']} {row[opt]}" for opt in OPTIONS]
    opt_embs  = normalize(st_model.encode(opt_texts, convert_to_numpy=True, show_progress_bar=False))
    sims_st   = (opt_embs @ train_embeddings_full.T).max(axis=1)
    st_r      = [OPTIONS[i] for i in np.argsort(-sims_st)]

    # MLP
    sims  = list(scores_tf.values())
    diffs = [sims[i] - sims[j] for i in range(5) for j in range(i + 1, 5)]
    x     = np.array(sims + diffs, dtype=np.float32).reshape(1, -1)
    proba = mlp_full.predict_proba(x)[0]
    mlp_r = [OPTIONS[j] for j in np.argsort(-proba)]

    # Weighted Borda voting
    return borda_ensemble([tfidf_r] * 2 + [st_r] * 3 + [mlp_r] * 2)[:3]


print("Generating submission...")
rows = []
for i, (_, row) in enumerate(test_df_clean.iterrows()):
    pred = predict_test_row(row)
    rows.append({"ID": row["id"], "Prediction": " ".join(pred)})
    if (i + 1) % 100 == 0:
        print(f"{i+1}/{len(test_df_clean)}")

submission = pd.DataFrame(rows)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")
print(submission.head(10).to_string(index=False))

In [ ]:
# Validate submission

errors = []

for _, row in submission.iterrows():
    preds = str(row["Prediction"]).split()

    if (
        len(preds) != 3
        or len(set(preds)) != 3
        or not all(p in OPTIONS for p in preds)
    ):
        errors.append(row["ID"])

if errors:
    print(f"Invalid rows: {len(errors)}")
else:
    print("Submission validation passed.")

print("\nTop-1 Prediction Distribution")
print(
    submission["Prediction"]
    .str.split()
    .str[0]
    .value_counts()
    .sort_index()
)